<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<span style="display:inline-block;margin-bottom:10px;padding:2px 12px;border-radius:999px;background:#6366f1;color:#fff;font:600 11px/20px system-ui,sans-serif;letter-spacing:.05em;text-transform:uppercase">Step 1</span>
<h1 style="font-size:1.35em;margin:.3em 0 .5em;line-height:1.3;font-weight:700;border:0;padding:0">Step 1: Settings</h1>
<p style="margin:.55em 0">Model, voice, turn detection, token caps, limits, prompts, which tools each agent may use: all of it lives in
<b>config</b>, not code. Three rules make that safe in production:</p>
<ul style="margin:.4em 0;padding-left:1.4em">
  <li style="margin:.2em 0">one file (<code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">$APP_CONFIG</code> or <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">config/app.toml</code>), any value overridable by an environment variable;</li>
  <li style="margin:.2em 0"><b>strict</b>: an unknown key or a wrong type stops the server at startup (a typo must never be silently ignored);</li>
  <li style="margin:.2em 0">secrets only from the environment.</li>
</ul>

</div>

<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<h2 style="font-size:1.2em;margin:.3em 0 .5em;line-height:1.3;font-weight:700;border:0;padding:0">The settings module</h2><p style="margin:.55em 0">Imports, the project root, and one error type for every settings problem.</p>

</div>

In [1]:
%%writefile build/app/settings.py
"""Settings: one TOML file + environment overrides, checked once at startup.

Every tunable lives in config, not code: model, voice, turn detection, token caps, limits, prompts, agents.

- File: $APP_CONFIG, else config/app.toml.
- Override any value with APP__<SECTION>__<KEY> (values parsed as JSON when they can be), e.g.
  APP__SERVER__PORT=9000  APP__OPENAI__MODEL=gpt-realtime  APP__AGENTS__FRONT_DESK__TOOLSETS='["identity"]'
- Secrets only from the environment (OPENAI_API_KEY). `openai.dev_key_ini` is a development convenience
  that production config leaves empty.
- Unknown keys and wrong types are errors: a typo must stop the server, not be silently ignored.
"""
import configparser
import json
import os
import tomllib
from dataclasses import MISSING, dataclass, field, fields
from pathlib import Path
from typing import Mapping

ROOT = Path(__file__).resolve().parent.parent


class SettingsError(ValueError):
    pass


Overwriting build/app/settings.py


<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<p style="margin:.55em 0">Each config section is a frozen dataclass: the field name is the key, the type annotation is the check, the default
is the default. Frozen: nothing can change a setting after startup.</p>

</div>

In [2]:
%%writefile -a build/app/settings.py

@dataclass(frozen=True)
class Server:
    host: str = "127.0.0.1"
    port: int = 8000
    origins: tuple = ("http://127.0.0.1:8000", "http://localhost:8000")  # browsers allowed to open /ws
    max_sessions: int = 20          # concurrent conversations; more get a "busy" answer
    session_limit_s: int = 300      # hard cap per conversation (bounds paid usage)
    max_message_bytes: int = 65536  # largest browser frame accepted
    drain_s: int = 10               # on shutdown, how long running conversations may finish
    log_format: str = "text"        # text | json
    log_level: str = "INFO"


@dataclass(frozen=True)
class OpenAI:
    url: str = "wss://api.openai.com/v1/realtime"
    model: str = "gpt-realtime-mini"
    voice: str = "marin"
    transcribe_model: str = "gpt-4o-mini-transcribe"
    max_output_tokens: int = 1000   # per response; audio counts (~300 tokens = ~10 s of speech)
    dev_key_ini: str = ""           # dev only: an ini file with [dev] openai_key


@dataclass(frozen=True)
class Resilience:
    stall_s: float = 5.0            # quiet this long mid-response -> ping
    ping_timeout_s: float = 5.0     # no pong in this long -> the connection is dead: reconnect
    max_resumes: int = 2            # reconnects per conversation


@dataclass(frozen=True)
class Clinic:
    name: str = "City Care Clinic"


@dataclass(frozen=True)
class Store:
    path: str = "data/app.db"
    seed_demo: bool = True          # demo doctors/patients on an empty database (never in production)


@dataclass(frozen=True)
class Otp:
    ttl_s: int = 300
    max_tries: int = 3
    max_sends: int = 3              # codes texted per conversation (stops SMS pumping)


@dataclass(frozen=True)
class Skills:
    timeout_s: float = 10.0         # a skill slower than this returns an error instead of hanging the call
    filler_audio: str = "assets/filler_marin.pcm"  # played the instant a tool call starts; "" = none


Appending to build/app/settings.py


<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<p style="margin:.55em 0">An <b>agent</b> is a prompt file plus the skill modules (toolsets) it may use, its own turn detection, and whether to
play a filler while tools run. One server can host several agents.</p>

</div>

In [3]:
%%writefile -a build/app/settings.py

@dataclass(frozen=True)
class Agent:
    prompt: str                     # template file, relative to the project root
    toolsets: tuple                 # skill modules (app/skills/<name>.py) this agent may use; "*" = all
    vad: dict = field(default_factory=lambda: {"type": "server_vad", "silence_duration_ms": 500})
    filler: bool = True


@dataclass(frozen=True)
class Settings:
    server: Server
    openai: OpenAI
    resilience: Resilience
    clinic: Clinic
    store: Store
    otp: Otp
    skills: Skills
    agents: dict
    root: Path = ROOT

    def path(self, p: str) -> Path:
        """A configured path, relative to the project root unless absolute."""
        return Path(p) if Path(p).is_absolute() else self.root / p


Appending to build/app/settings.py


<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<h2 style="font-size:1.2em;margin:.3em 0 .5em;line-height:1.3;font-weight:700;border:0;padding:0">Strict parsing</h2>
<p style="margin:.55em 0"><code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">_value</code> checks one value against its annotation (careful: <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">True</code> is an <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">int</code> in Python,
so integers are checked exactly). <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">_build</code> turns a TOML table into a dataclass and rejects unknown keys.
<code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">_apply_env</code> merges <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">APP__SECTION__KEY</code> variables, parsing JSON where it can.</p>

</div>

In [4]:
%%writefile -a build/app/settings.py

SECTIONS = {"server": Server, "openai": OpenAI, "resilience": Resilience, "clinic": Clinic, "store": Store,
            "otp": Otp, "skills": Skills}


def _value(v, kind, where):
    if kind is tuple and isinstance(v, list):
        v = tuple(v)
    ok = (isinstance(v, (int, float)) and not isinstance(v, bool)) if kind is float else \
        (type(v) is int) if kind is int else isinstance(v, kind)  # bool is an int: check exactly
    if not ok:
        raise SettingsError(f"{where}: expected {kind.__name__}, got {type(v).__name__}")
    return float(v) if kind is float else v


def _build(cls, data, where):
    if not isinstance(data, dict):
        raise SettingsError(f"{where}: expected a table")
    known = {f.name: f for f in fields(cls)}
    if unknown := sorted(set(data) - set(known)):
        raise SettingsError(f"{where}: unknown key(s) {unknown}")
    kw = {}
    for name, f in known.items():
        if name in data:
            kw[name] = _value(data[name], f.type, f"{where}.{name}")
        elif f.default is MISSING and f.default_factory is MISSING:
            raise SettingsError(f"{where}: missing required key '{name}'")
    return cls(**kw)


def _apply_env(data: dict, env: Mapping[str, str]) -> None:
    for key, raw in env.items():
        if not key.startswith("APP__"):
            continue
        path = [p.lower() for p in key[5:].split("__")]
        try:
            value = json.loads(raw)
        except ValueError:
            value = raw
        node = data
        for p in path[:-1]:
            node = node.setdefault(p, {})
            if not isinstance(node, dict):
                raise SettingsError(f"{key}: {p} is not a table")
        node[path[-1]] = value


Appending to build/app/settings.py


<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<p style="margin:.55em 0"><code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">load</code> ties it together and also checks what only makes sense as a whole: every agent's prompt file exists,
no agent has an empty toolset. <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">api_key</code> reads the secret; the dev fallback is an ini file path that production
config leaves empty.</p>

</div>

In [5]:
%%writefile -a build/app/settings.py

def load(path: str | Path | None = None, env: Mapping[str, str] = os.environ) -> Settings:
    path = Path(path or env.get("APP_CONFIG") or ROOT / "config" / "app.toml")
    try:
        data = tomllib.loads(path.read_text(encoding="utf-8"))
    except (OSError, tomllib.TOMLDecodeError) as e:
        raise SettingsError(f"cannot read settings {path}: {e}") from None
    _apply_env(data, env)
    if unknown := sorted(set(data) - set(SECTIONS) - {"agents"}):
        raise SettingsError(f"unknown section(s) {unknown}")
    sections = {name: _build(cls, data.get(name, {}), name) for name, cls in SECTIONS.items()}
    agents = {name: _build(Agent, spec, f"agents.{name}") for name, spec in data.get("agents", {}).items()}
    if not agents:
        raise SettingsError("no [agents.<name>] defined")
    s = Settings(**sections, agents=agents)
    for name, a in agents.items():
        if not s.path(a.prompt).is_file():
            raise SettingsError(f"agents.{name}.prompt: no file {s.path(a.prompt)}")
        if not a.toolsets:
            raise SettingsError(f"agents.{name}.toolsets: empty")
    if s.server.log_format not in ("text", "json"):
        raise SettingsError("server.log_format: text or json")
    return s


def api_key(s: Settings, env: Mapping[str, str] = os.environ) -> str:
    """OPENAI_API_KEY from the environment; in development optionally from openai.dev_key_ini. Never logged."""
    if key := env.get("OPENAI_API_KEY"):
        return key
    if s.openai.dev_key_ini:
        c = configparser.ConfigParser(interpolation=None)
        if c.read(s.openai.dev_key_ini) and c.has_option("dev", "openai_key"):
            return c["dev"]["openai_key"].strip()
    raise SettingsError("no OPENAI_API_KEY in the environment (and no usable openai.dev_key_ini)")


Appending to build/app/settings.py


<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<h2 style="font-size:1.2em;margin:.3em 0 .5em;line-height:1.3;font-weight:700;border:0;padding:0">The config file and the prompts it points to</h2>
<p style="margin:.55em 0">Development defaults. Note the two agents and their different turn detection: dictating details has pauses, so the
front desk waits 900 ms of silence before answering.</p>

</div>

In [6]:
%%writefile build/config/app.toml
# Development settings. Production: config/production.toml (APP_CONFIG=config/production.toml).
# Any value can be overridden by an environment variable APP__<SECTION>__<KEY>, e.g. APP__SERVER__PORT=9000.

[server]
host = "127.0.0.1"
port = 8000
origins = ["http://127.0.0.1:8000", "http://localhost:8000"]
max_sessions = 20
session_limit_s = 300
log_format = "text"
log_level = "INFO"

[openai]
model = "gpt-realtime-mini"
voice = "marin"
transcribe_model = "gpt-4o-mini-transcribe"
max_output_tokens = 1000
dev_key_ini = 'D:\abhinay\DocAssist\pyDocAssist\config.ini'   # dev only; production uses OPENAI_API_KEY

[resilience]
stall_s = 5
ping_timeout_s = 5
max_resumes = 2

[clinic]
name = "City Care Clinic"

[store]
path = "data/app.db"
seed_demo = true

[otp]
ttl_s = 300
max_tries = 3
max_sends = 3

[skills]
timeout_s = 10
filler_audio = "assets/filler_marin.pcm"

# An agent = a prompt template + the skill modules (toolsets) it may use + its turn detection.
# A new skill in an existing toolset is picked up automatically; a new toolset is one name added here.
[agents.front_desk]
prompt = "prompts/front_desk.md"
toolsets = ["identity", "registration", "appointments", "clinic_info"]
vad = { type = "server_vad", silence_duration_ms = 900 }   # dictation has pauses: don't cut callers off

[agents.support]
prompt = "prompts/support.md"
toolsets = ["orders"]
vad = { type = "server_vad", silence_duration_ms = 500 }


Overwriting build/config/app.toml


<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<p style="margin:.55em 0">Prompts are files too, so they can be tuned without touching code. <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">{clinic}</code>, <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">{today}</code> and
<code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">{skills}</code> are filled in per conversation (step 5). Each rule here was earned in testing: models announce
actions instead of calling tools, forget what the caller asked after verifying, re-check slots after a "yes".</p>

</div>

In [7]:
%%writefile build/prompts/front_desk.md
You are the voice front desk of {clinic}. Today is {today}.
Be brief and warm. Ask one thing at a time.
Speak the caller's language: reply in the language they use (English, Hindi, Tamil, or a mix) and switch when they switch. Tool arguments stay in English: names in English letters, numbers as digits.

Rules:
- Use the tools for every fact: never invent doctors, times, codes, appointments or patient details.
- When a tool result has a "next", do what it says.
- When you have what a tool needs, call it in that same turn. Never announce an action ("I'll send a code", "I'll book that") instead of calling the tool.
- Anything about an existing patient (their appointments, booking, cancelling) needs a verified caller: ask for their mobile number, call verify_patient, then confirm_code with the code they read from the SMS. A new patient becomes verified by registering.
- The moment the caller gives a mobile number, call verify_patient in that same turn. Codes are sent only by verify_patient or start_registration: never say a code was or will be sent unless one returned code_sent.
- Copy phone numbers and codes digit for digit as the caller said them.
- Give tools dates as YYYY-MM-DD, worked out from today's date ("tomorrow", "next Tuesday"), and times like 9:15 AM.
- Before booking or cancelling, say what you are about to do (doctor, day, time) and get a yes. Once they say yes, call book_appointment (or cancel_appointment) right away: do not check slots again or ask again. If the time they asked for is not free, offer the nearest free times instead.

Tools:
{skills}


Overwriting build/prompts/front_desk.md


In [8]:
%%writefile build/prompts/support.md
You are a support agent for an online shop. Today is {today}. Be brief.
Use the tools for orders and temperatures; never invent order details.

Tools:
{skills}


Overwriting build/prompts/support.md


<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<h2 style="font-size:1.2em;margin:.3em 0 .5em;line-height:1.3;font-weight:700;border:0;padding:0">Try it</h2>

</div>

In [9]:
import sys
sys.path.insert(0, 'build')   # `import app` now means the app you are building

In [10]:
from app import settings
s = settings.load()
s.openai.model, s.server.port, list(s.agents), s.agents['front_desk'].vad

('gpt-realtime-mini',
 8000,
 ['front_desk', 'support'],
 {'type': 'server_vad', 'silence_duration_ms': 900})

<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<p style="margin:.55em 0">Environment overrides win, with JSON parsing:</p>

</div>

In [11]:
s2 = settings.load(env={'APP__SERVER__PORT': '9000', 'APP__AGENTS__SUPPORT__TOOLSETS': '["orders", "extra"]'})
s2.server.port, s2.agents['support'].toolsets

(9000, ('orders', 'extra'))

<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<p style="margin:.55em 0">Typos and wrong types stop the server:</p>

</div>

In [12]:
for env in ({'APP__SERVER__PROT': '9000'}, {'APP__SERVER__PORT': '"9000"'}, {'APP__STORE__SEED_DEMO': '1'}):
    try:
        settings.load(env=env)
    except settings.SettingsError as e:
        print('refused:', e)

refused: server: unknown key(s) ['prot']
refused: server.port: expected int, got str
refused: store.seed_demo: expected bool, got int


In [13]:
settings.api_key(s, env={'OPENAI_API_KEY': 'sk-test'})

'sk-test'

<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<p style="margin:.55em 0"><b>Next:</b> <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">02_skill_registry.ipynb</code>: turning plain functions into tools.</p>

</div>